# 🚗 AURA DRIVE
## AI-Powered Adaptive Driver Intelligence & Safety System
> Run each cell in order. Allow camera access when prompted.

In [ ]:
# ── Cell 1 — Install libraries + download dlib model (~100 MB) ──
!pip install -q opencv-python-headless dlib imutils scipy numpy pandas matplotlib ipython

import urllib.request, bz2, os
MODEL_URL  = 'http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2'
MODEL_BZ2  = 'shape_predictor_68_face_landmarks.dat.bz2'
MODEL_FILE = 'shape_predictor_68_face_landmarks.dat'

if not os.path.exists(MODEL_FILE):
    print('Downloading dlib 68-point model...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_BZ2)
    with bz2.open(MODEL_BZ2) as f_in, open(MODEL_FILE, 'wb') as f_out:
        f_out.write(f_in.read())
    os.remove(MODEL_BZ2)
    print('Model ready')
else:
    print('Model already present')

In [ ]:
# ── Cell 2 — Clone repo from GitHub + load all modules (FIXED) ──
import sys, os

REPO_ROOT = '/content/aura_drive'

# Clone from your GitHub repo
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/vishal9519-vis/AURA-DRIVE.git /content/aura_drive
else:
    print('Repo already cloned, pulling latest...')
    !cd /content/aura_drive && git pull

# Add repo root to Python path
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Change working directory so relative paths work
os.chdir(REPO_ROOT)

# Move dlib model into repo root if it was downloaded to /content
MODEL_FILE = 'shape_predictor_68_face_landmarks.dat'
if os.path.exists(f'/content/{MODEL_FILE}') and not os.path.exists(f'{REPO_ROOT}/{MODEL_FILE}'):
    import shutil
    shutil.copy(f'/content/{MODEL_FILE}', f'{REPO_ROOT}/{MODEL_FILE}')
    print('Copied dlib model into repo root')

# Import all modules
from config                  import Config
from modules.face_detection  import load_predictor, detect_faces, get_landmarks
from modules.eye_tracker     import get_ear, draw_eyes
from modules.fatigue         import get_mar, update_fatigue
from modules.head_pose       import estimate_head_pose, is_distracted
from modules.emotion         import detect_emotion
from modules.risk            import compute_risk, adaptive_response, update_attention
from modules.tracker         import AuraTracker
from dashboard.analytics     import generate_dashboard

load_predictor(os.path.join(REPO_ROOT, MODEL_FILE))
print('All modules loaded')

In [ ]:
# ── Cell 3 — Initialise AuraTracker session ──
tracker = AuraTracker()
print('Session initialised')
print(f'EAR threshold : {Config.EAR_THRESHOLD}')
print(f'MAR threshold : {Config.MAR_THRESHOLD}')
print(f'Consec frames : {Config.CONSEC_FRAMES}')

In [ ]:
# ── Cell 4 — Define webcam bridge + main analyse function ──
import cv2, time
import numpy as np
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode

def take_photo():
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise(r => setTimeout(r, 500));
            const canvas = document.createElement('canvas');
            canvas.width  = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data   = eval_js(f'takePhoto({Config.WEBCAM_QUALITY})')
    binary = b64decode(data.split(',')[1])
    arr    = np.frombuffer(binary, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


def analyse_frame(frame, tracker):
    """Run the full AURA DRIVE pipeline on a single frame."""
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detect_faces(gray)

    if len(faces) == 0:
        tracker.frame_count += 1
        print(f'  Frame {tracker.frame_count:>3} | [!] No face detected — skipped')
        return

    face = faces[0]
    lm   = get_landmarks(gray, face)

    ear, left_eye, right_eye = get_ear(lm)
    mar, mouth               = get_mar(lm)
    yaw, pitch, roll         = estimate_head_pose(lm, frame.shape)
    distracted               = is_distracted(yaw, pitch)

    attention = update_attention(tracker.attention_score, ear, distracted, mar)
    emotion   = detect_emotion(ear, mar, tracker.fatigue_level, attention)
    fatigue   = update_fatigue(tracker.fatigue_level, ear, mar, emotion, distracted)
    risk      = compute_risk(attention, fatigue, emotion, distracted)
    level, msg, color = adaptive_response(risk)

    tracker.update(ear, mar, yaw, pitch, roll,
                   emotion, attention, fatigue, risk, level)

    print(f'  Frame {tracker.frame_count:>3} | '
          f'EAR={ear:.3f}  MAR={mar:.3f}  '
          f'Yaw={yaw:+.1f}  Pitch={pitch:+.1f}  '
          f'Emotion={emotion:<8}  '
          f'Attention={attention:5.1f}  '
          f'Fatigue={fatigue:5.1f}  '
          f'Risk={risk:5.1f}  '
          f'-> {level}')

print('analyse_frame() defined')

In [ ]:
# ── Cell 5 — Single photo test ──
print('Click Allow for camera access...')
frame = take_photo()
print(f'Frame captured: {frame.shape}')
analyse_frame(frame, tracker)

In [ ]:
# ── Cell 6 — Live 10-frame session ──
import time
print('Starting 10-frame session...')
for i in range(10):
    print(f'\nCapturing frame {i+1}/10...')
    frame = take_photo()
    analyse_frame(frame, tracker)
    time.sleep(Config.CAPTURE_DELAY)
print('\n10-frame session complete')

In [ ]:
# ── Cell 7 — Analytics dashboard ──
fig = generate_dashboard(tracker, save_path='aura_drive_dashboard.png')
from IPython.display import Image as IPImage
display(IPImage('aura_drive_dashboard.png'))
tracker.summary()

In [ ]:
# ── Cell 8 — Full Phase 4 session (15 frames) ──
import time
tracker2 = AuraTracker()
print('Starting full 15-frame Phase 4 session...')
for i in range(15):
    print(f'\nCapturing frame {i+1}/15...')
    frame = take_photo()
    analyse_frame(frame, tracker2)
    time.sleep(Config.CAPTURE_DELAY)

fig2 = generate_dashboard(tracker2, save_path='aura_drive_phase4_dashboard.png')
from IPython.display import Image as IPImage
display(IPImage('aura_drive_phase4_dashboard.png'))
tracker2.summary()